# 02 — Model Training, Hyperparameter Tuning & Evaluation

Trains Logistic Regression and XGBoost baselines, tunes each
(`RandomizedSearchCV` for LR, Optuna/TPE for XGBoost), and evaluates both on
the held-out test set with ROC-AUC, PR-AUC, F1, KS statistic, and Brier
score (PR-AUC/Brier included because Home Credit's default rate is
imbalanced — ROC-AUC alone can be misleading here).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from sklearn.model_selection import train_test_split

from dac.config import CONFIG
from dac.data.loader import load_home_credit
from dac.features.engineering import build_preprocessor, engineer_home_credit_features, split_feature_columns
from dac.models.train import build_logistic_regression_pipeline, build_xgboost_pipeline, compute_scale_pos_weight, fit
from dac.models.tune import tune_logistic_regression, tune_xgboost
from dac.models.evaluate import compute_metrics, plot_evaluation_suite, compare_models

hc_cfg = CONFIG["data"]["home_credit"]
df, _ = load_home_credit()
df = engineer_home_credit_features(df)

exclude_cols = [hc_cfg["id_col"], *hc_cfg["protected_attributes"]]
numeric_cols, categorical_cols = split_feature_columns(df, hc_cfg["target_col"], exclude_cols)
X, y = df[numeric_cols + categorical_cols], df[hc_cfg["target_col"]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=CONFIG["seed"], stratify=y)
preprocessor = build_preprocessor(numeric_cols, categorical_cols)
print(X_train.shape, X_test.shape, y_train.mean(), y_test.mean())

## Baselines

In [ ]:
lr_baseline = fit(build_logistic_regression_pipeline(preprocessor), X_train, y_train, "logistic_regression_baseline")
xgb_baseline = fit(
    build_xgboost_pipeline(preprocessor, scale_pos_weight=compute_scale_pos_weight(y_train)),
    X_train, y_train, "xgboost_baseline",
)

baseline_results = {}
for trained in (lr_baseline, xgb_baseline):
    proba = trained.pipeline.predict_proba(X_test)[:, 1]
    baseline_results[trained.name] = compute_metrics(y_test.to_numpy(), proba)
    plot_evaluation_suite(y_test.to_numpy(), proba, trained.name, CONFIG["paths"]["figures_dir"] / "home_credit")
compare_models(baseline_results, CONFIG["paths"]["metrics_dir"] / "baseline")

## Hyperparameter tuning

In [ ]:
lr_tuned = tune_logistic_regression(preprocessor, X_train, y_train, n_iter=15, cv_folds=5, seed=CONFIG["seed"])
xgb_tuned = tune_xgboost(preprocessor, X_train, y_train, n_trials=CONFIG["tuning"]["n_trials"], cv_folds=5, seed=CONFIG["seed"])
lr_tuned.pipeline.fit(X_train, y_train)
xgb_tuned.pipeline.fit(X_train, y_train)
print("LR best:", lr_tuned.best_params, lr_tuned.best_cv_score)
print("XGB best:", xgb_tuned.best_params, xgb_tuned.best_cv_score)

In [ ]:
tuned_results = {}
for name, pipeline in [("logistic_regression_tuned", lr_tuned.pipeline), ("xgboost_tuned", xgb_tuned.pipeline)]:
    proba = pipeline.predict_proba(X_test)[:, 1]
    tuned_results[name] = compute_metrics(y_test.to_numpy(), proba)
    plot_evaluation_suite(y_test.to_numpy(), proba, name, CONFIG["paths"]["figures_dir"] / "home_credit")

all_results = {**baseline_results, **tuned_results}
compare_models(all_results, CONFIG["paths"]["metrics_dir"])

Continue to `03_explainability_shap.ipynb` for SHAP on the best tuned model, and `04_fairness_audit_mitigation.ipynb` for the fairness audit + reweighing mitigation.